# Why the holdout scores above the leaderboard (hypotheses H1 and H2, CPU)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default `MODEL = "r34"` no cell needs editing. To analyze another model, change only `MODEL` in the settings cell and attach its cache:

| `MODEL` | Encoder | Save the training notebook as | Checkpoint |
| --- | --- | --- | --- |
| `r34` (default) | `resnet34` | `train-unet-r34-crops-80ep` | `unet_r34_crops.pt` |
| `r50` | `resnet50` | `train-unet-r50-crops` | `unet_r50_crops.pt` |
| `effb3` | `efficientnet-b3` | `train-unet-effb3-crops` | `unet_effb3_crops.pt` |
| `r34_v2` | `resnet34` | `train-unet-r34-v2` | `unet_r34_v2.pt` |
| `r34_v2_agree` | `resnet34` | `train-unet-r34-v2-agree` | `unet_r34_v2_agree.pt` |

`docs/ROADMAP_V2.md` section 3 gives three hypotheses for a holdout PQ of about 0.41 against a public leaderboard of 0.34. This notebook measures two of them from the maps of `notebooks/export_probs.ipynb`, without a GPU:

- **H1, selection.** The settings were chosen on the same 142 files they are scored on. A cross-fit chooses on some dates and scores on the others, over 5 partitions of the dates. The sweep's maximum minus the cross-fit estimates what selection adds.
- **H2, annotator mix.** The holdout has 1.63 views per file, and one source estimates 2.44 for the test set [VERIFY]. The PQ is scored on files with one view, on files with several, and with the file mix tilted to 2.44 views per file.

It reads only the holdout side of the cache. It does not train, does not read the test images, does not write `submission.csv` and does not submit to Kaggle. These numbers say how much of the gap each hypothesis can account for. They do not prove a cause.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip), a version that contains `filament.gap_analysis`. An older version stops at the imports.
3. The private dataset `solar-probs-<MODEL>`, made from the output of `export_probs.ipynb` for this model.

**Settings**

- Accelerator: **None**. This notebook does not use a GPU.
- Internet: **Off** is enough.

**Outputs** (in `/kaggle/working`)

- `holdout_gap_<MODEL>.csv`: every number below.
- `holdout_per_file_<MODEL>.csv`: per-file counts at the current setting, for later paired comparisons.

Both are small and hold no image or mask data, so they can go in `results/`. The running time is not measured yet [VERIFY]: it scores 294 settings on each of 142 frames, on one CPU core, and a progress line is printed every 10 frames.

## 1. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Settings and the cache

- `CURRENT` is the setting of the holdout 0.409 and leaderboard 0.34 (`docs/PROJECT_REPORT.md`, row 12).
- `TARGET_VIEWS` is one source's estimate of views per test image, not verified. It feeds a what-if, not a measurement of the test set.
- The grid and the partitions come from `filament.gap_analysis`.
- The cache has to come from exactly this checkpoint: its manifest goes through the same guard as a weights file. Leave `ALLOW_UNVERIFIED_WEIGHTS` as `False`.

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.checkpoint import crop_checkpoint, verify_and_report
from filament.crop_data import read_gray
from filament.gap_analysis import (
    CROSSFIT_FOLDS,
    CROSSFIT_SEEDS,
    CURRENT_POINT,
    TEST_VIEWS_PER_FILE,
    format_summary,
    gap_points,
    gap_summary_rows,
    holdout_dates,
    per_file_rows,
    write_per_file_rows,
    write_summary_rows,
)
from filament.probcache import (
    ProbCache,
    find_cache_root,
    manifest_fingerprint,
    read_manifest,
    split_dir,
)
from filament.sweep import count_views, per_file_counts
from filament.train_data import split_training_images

# ---- Which model: a name of filament.checkpoint.CROP_CHECKPOINTS --------------
MODEL = "r34"
EXPECTED = crop_checkpoint(MODEL)

# ---- The analysis ------------------------------------------------------------
CURRENT = CURRENT_POINT
TARGET_VIEWS = TEST_VIEWS_PER_FILE
ALLOW_UNVERIFIED_WEIGHTS = False

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
SUMMARY_PATH = OUTPUT_DIR / f"holdout_gap_{MODEL}.csv"
PER_FILE_PATH = OUTPUT_DIR / f"holdout_per_file_{MODEL}.csv"

CACHE_ROOT = find_cache_root(MODEL, INPUT_ROOT)
print(f"cache={CACHE_ROOT}")
manifest = read_manifest(CACHE_ROOT, MODEL)
print(manifest)
verify_and_report(
    manifest_fingerprint(manifest),
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=EXPECTED,
)

## 3. Date holdout

The same split as the training notebook: the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). Every holdout file has to be in the cache.

In [ ]:
IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
holdout = split_training_images(loaded.images).holdout
holdout_views = count_views(holdout)
print(f"holdout files={len(holdout)} annotator views={holdout_views}")
if (len(holdout), holdout_views) != (142, 231):
    print(
        "WARNING: the holdout is not 142 files with 231 views, the numbers the "
        "training notebook used. [VERIFY] the data version."
    )
cache = ProbCache(split_dir(CACHE_ROOT, MODEL, "holdout"))
missing = [image.file_name for image in holdout if image.file_name not in cache]
if missing:
    raise RuntimeError(
        f"{len(missing)} holdout files are not in the cache, for example {missing[0]}."
    )

## 4. Score every setting, file by file

The scoring is `filament.sweep`: pooled per-view PQ (`fast_view_counts`, `PqCounts`), kept per file. The maps are the flip-TTA means from the cache. Nothing goes through a network.

In [ ]:
import time


def _read(image):
    return read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))


def _load(image):
    return cache.mean(image.file_name)


def _progress(index, total, image):
    if index == 1 or index % 10 == 0 or index == total:
        print(f"  {index}/{total} {image.file_name}", flush=True)


POINTS = gap_points()
if CURRENT not in POINTS:
    raise ValueError(f"{CURRENT} is not on the grid")
print(f"{len(POINTS)} settings on {len(holdout)} files")
begin = time.monotonic()
per_file = per_file_counts(holdout, _read, _load, POINTS, progress=_progress)
print(f"scoring took {(time.monotonic() - begin) / 60:.1f} min")
print(f"pooled PQ at {CURRENT}: {per_file.pooled(CURRENT).pq:.4f}")

## 5. The numbers

How to read them, against the gap of about 0.07 between holdout and leaderboard:

- `current_setting` is the PQ at `CURRENT` with a 95% bootstrap interval over files. For `r34` it should be about 0.409.
- `h2_stratum_one_view` and `h2_stratum_several_views`: if the second is near 0.34, the mix of annotators can account for most of the gap. Those two groups of files may also differ in other ways.
- `h2_reweighted_views`: the PQ with the file mix tilted to `TARGET_VIEWS` views per file, with how many effective files that leaves.
- `h1_crossfit_mean`: the PQ of settings chosen on other dates. `h1_sweep_maximum_minus_crossfit` estimates what selection adds. Each fold chooses on half the files, so a choice there is noisier than one on all 142, and this leans high.

Add one row to `docs/RESULTS.md` from this table, and put both CSV files in `results/`.

In [ ]:
dates = holdout_dates(holdout)
rows = gap_summary_rows(
    per_file,
    dates,
    point=CURRENT,
    target_views=TARGET_VIEWS,
    seeds=CROSSFIT_SEEDS,
    n_folds=CROSSFIT_FOLDS,
)
print(format_summary(rows))
write_summary_rows(SUMMARY_PATH, rows)
write_per_file_rows(PER_FILE_PATH, per_file_rows(per_file, CURRENT))
print(f"wrote {SUMMARY_PATH.name} and {PER_FILE_PATH.name} in {OUTPUT_DIR}")
print("This notebook read no test image, wrote no submission and did not submit.")